# Multi-Model Excess Return CSV Builder

**Objective:** For each stock (GME, AMC) and each bar frequency (1-min, 5-min, 10-min),
construct a unified CSV containing:
- Raw price and bar return
- SPY return and simple market-adjusted excess return
- Fama-French factors (MKT_RF, SMB, HML, MOM)
- Abnormal returns and cumulative abnormal returns under three models:
  - **CAPM**: `R = α + β·SPY + ε`
  - **FF3**: `R = α + β_MKT·MKT_RF + β_SMB·SMB + β_HML·HML + ε`
  - **FF4**: `R = α + β_MKT·MKT_RF + β_SMB·SMB + β_HML·HML + β_MOM·MOM + ε`

Models are estimated over a **120-day pre-event estimation window** (ending Jan 26, 2021)
**at each bar frequency** (betas estimated on 5-min bars for 5-min residuals, etc.).

**Output:** 6 CSV files — `{SYMBOL}-{freq}-multi-model-excess-return.csv`

---
## 1. Setup

In [1]:
import pathlib
import warnings
from datetime import timedelta

import numpy as np
import pandas as pd
import statsmodels.api as sm

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_DIR     = PROJECT_ROOT / "Minute price and exceed return" / "output_data"
FF_DIR       = PROJECT_ROOT / "FF-Factors"
OUTPUT_DIR   = DATA_DIR  # save alongside existing CSVs

# ── Stock definitions ──────────────────────────────────────────────────────────
STOCKS = {
    "GME": {
        "excess_return_file": "GME-minute_price-excess-return.csv",
        "price_col":  "gme_price",
        "return_col": "gme_return",
    },
    "AMC": {
        "excess_return_file": "AMC-minute_price-excess-return.csv",
        "price_col":  "amc_price",
        "return_col": "amc_return",
    },
}

# ── Bar frequencies ────────────────────────────────────────────────────────────
FREQS = {
    "1min":  "1T",
    "5min":  "5T",
    "10min": "10T",
}

# ── Estimation window ──────────────────────────────────────────────────────────
# [-120, -2]: end 2 days before first outage (matching FF4 notebook)
ESTIMATION_WINDOW_DAYS = 120
ESTIMATION_LAG_DAYS    = 2
FIRST_OUTAGE_START     = pd.Timestamp("2021-01-27 11:29:00")
ESTIMATION_END         = FIRST_OUTAGE_START - timedelta(days=ESTIMATION_LAG_DAYS)

print(f"Estimation window end : {ESTIMATION_END}")
print(f"Estimation window start: {ESTIMATION_END - timedelta(days=ESTIMATION_WINDOW_DAYS)}")
print(f"Output directory      : {OUTPUT_DIR}")

Estimation window end : 2021-01-25 11:29:00
Estimation window start: 2020-09-27 11:29:00
Output directory      : /Users/rajvardhan/Desktop/Projects/RA-MemeStocks/Minute price and exceed return/output_data


---
## 2. Load 1-Minute Data

In [2]:
def load_1min_stock(symbol):
    """Load the existing 1-min excess-return CSV for a stock."""
    path = DATA_DIR / STOCKS[symbol]["excess_return_file"]
    df = (
        pd.read_csv(path, parse_dates=["datetime"])
        .drop_duplicates("datetime")
        .sort_values("datetime")
        .set_index("datetime")
    )
    return df


def load_ff_factors():
    """Load minute-level Fama-French four factors (trading hours only)."""
    path = FF_DIR / "ff_factors_20201101_20210430_minute.csv"
    ff = (
        pd.read_csv(path, parse_dates=["datetime"])
        .drop_duplicates("datetime")
        .sort_values("datetime")
        .set_index("datetime")
    )
    return ff


gme_1min = load_1min_stock("GME")
amc_1min = load_1min_stock("AMC")
ff_1min  = load_ff_factors()

print(f"GME 1-min: {len(gme_1min):,} rows | {gme_1min.index.min()} → {gme_1min.index.max()}")
print(f"AMC 1-min: {len(amc_1min):,} rows | {amc_1min.index.min()} → {amc_1min.index.max()}")
print(f"FF  1-min: {len(ff_1min):,} rows  | {ff_1min.index.min()} → {ff_1min.index.max()}")

GME 1-min: 99,878 rows | 2020-11-02 04:01:00 → 2021-04-30 19:59:00
AMC 1-min: 105,370 rows | 2020-11-02 04:00:00 → 2021-04-30 19:59:00
FF  1-min: 43,084 rows  | 2020-11-02 09:31:00 → 2021-04-30 16:00:00


---
## 3. Resampling Functions

Resample 1-min bars to 5-min and 10-min:
- **Price**: last trade price in bar
- **Returns**: compound minute returns within bar → `(1+r₁)(1+r₂)... − 1`
- **FF factors**: sum (log-return-based factors are additive); `min_count=1` preserves NaN for fully after-hours bars

In [3]:
def resample_stock(stock_1min, price_col, return_col, freq_alias):
    """
    Resample 1-min stock data to `freq_alias` (e.g. '5T').
    For 1T (1-min), returns the data as-is with renamed columns.
    """
    if freq_alias == "1T":
        out = stock_1min[[price_col, return_col, "spy_return"]].copy()
        out.columns = ["price", "return", "spy_return"]
        return out

    # Price: last trade in bar
    price = stock_1min[price_col].resample(freq_alias).last()

    # Returns: compound; mark empty bars (no trades) as NaN
    def compound(series, alias):
        bar = (1 + series.fillna(0)).resample(alias).prod() - 1
        count = series.resample(alias).count()
        bar[count == 0] = np.nan
        return bar

    ret = compound(stock_1min[return_col], freq_alias)
    spy = compound(stock_1min["spy_return"], freq_alias)

    return pd.DataFrame({"price": price, "return": ret, "spy_return": spy})


def resample_ff(ff_1min, freq_alias):
    """
    Resample 1-min FF factors to `freq_alias`.
    Uses sum (factors are log-return based); min_count=1 preserves NaN
    for bars where ALL minutes are after-hours (no factor data).
    """
    if freq_alias == "1T":
        return ff_1min[["MKT_RF", "SMB", "HML", "MOM"]].copy()
    return ff_1min[["MKT_RF", "SMB", "HML", "MOM"]].resample(freq_alias).sum(min_count=1)


print("Resample functions defined.")

Resample functions defined.


---
## 4. Model Estimation Functions

Estimate CAPM, FF3, FF4 on the estimation window **at the target bar frequency**.
Inner join with FF factors restricts estimation to regular trading hours (09:30–16:00 ET).

In [4]:
def estimate_models(bar_data, ff_bar, estimation_end,
                    window_days=ESTIMATION_WINDOW_DAYS):
    """
    Estimate CAPM, FF3, FF4 on bar-frequency data within the estimation window.

    Parameters
    ----------
    bar_data      : DataFrame with columns [price, return, spy_return]
    ff_bar        : DataFrame with columns [MKT_RF, SMB, HML, MOM]
    estimation_end: last timestamp of estimation window

    Returns
    -------
    (m_capm, m_ff3, m_ff4) : fitted OLS models
    """
    est_start = estimation_end - timedelta(days=window_days)

    # Stock returns in window
    w = bar_data.loc[est_start:estimation_end].copy()

    # Join FF factors — inner join keeps only bars with factor data (trading hours)
    w = w.join(ff_bar, how="inner")
    w = w.dropna(subset=["return", "spy_return", "MKT_RF", "SMB", "HML", "MOM"])

    y = w["return"]

    m_capm = sm.OLS(y, sm.add_constant(w[["spy_return"]])).fit()
    m_ff3  = sm.OLS(y, sm.add_constant(w[["MKT_RF", "SMB", "HML"]])).fit()
    m_ff4  = sm.OLS(y, sm.add_constant(w[["MKT_RF", "SMB", "HML", "MOM"]])).fit()

    return m_capm, m_ff3, m_ff4


def model_summary(symbol, freq_name, m_capm, m_ff3, m_ff4):
    """Print a compact parameter summary for all three models."""
    p4 = m_ff4.params
    print(f"  {symbol} @ {freq_name}  (n={int(m_capm.nobs):,} bars in estimation window)")
    print(f"    CAPM : α={m_capm.params.get('const',0):+.6f}  "
          f"β_mkt={m_capm.params.get('spy_return',0):.4f}  R²={m_capm.rsquared:.4f}  "
          f"σ={np.sqrt(m_capm.mse_resid):.6f}")
    print(f"    FF3  : α={m_ff3.params.get('const',0):+.6f}  "
          f"β_mkt={m_ff3.params.get('MKT_RF',0):.4f}  "
          f"β_smb={m_ff3.params.get('SMB',0):.4f}  "
          f"β_hml={m_ff3.params.get('HML',0):.4f}  R²={m_ff3.rsquared:.4f}")
    print(f"    FF4  : α={p4.get('const',0):+.6f}  "
          f"β_mkt={p4.get('MKT_RF',0):.4f}  "
          f"β_smb={p4.get('SMB',0):.4f}  "
          f"β_hml={p4.get('HML',0):.4f}  "
          f"β_mom={p4.get('MOM',0):.4f}  R²={m_ff4.rsquared:.4f}")


print("Model estimation functions defined.")

Model estimation functions defined.


---
## 5. Abnormal Return Computation

Apply estimated model parameters to the **full time series** to produce AR and CAR columns.

- **CAPM AR**: available all hours (uses spy_return as market proxy)
- **FF3/FF4 AR**: NaN for after-hours bars where factors are unavailable

In [5]:
def compute_ar_columns(bar_data, ff_bar, m_capm, m_ff3, m_ff4):
    """
    Compute expected returns, abnormal returns (AR), and cumulative AR (CAR)
    for all three models on the full bar time series.

    Returns a DataFrame with output columns only.
    """
    df = bar_data.join(ff_bar, how="left")  # left join: NaN for after-hours

    # ── CAPM (spy_return available all hours) ──────────────────────────────────
    p = m_capm.params
    df["expected_capm"] = p.get("const", 0) + p.get("spy_return", 0) * df["spy_return"]
    df["AR_capm"]       = df["return"] - df["expected_capm"]
    df["CAR_capm"]      = df["AR_capm"].cumsum()

    # ── FF3 (NaN after hours) ──────────────────────────────────────────────────
    p = m_ff3.params
    df["expected_ff3"] = (
        p.get("const",  0)
        + p.get("MKT_RF", 0) * df["MKT_RF"]
        + p.get("SMB",   0) * df["SMB"]
        + p.get("HML",   0) * df["HML"]
    )
    df["AR_ff3"]  = df["return"] - df["expected_ff3"]
    df["CAR_ff3"] = df["AR_ff3"].cumsum()

    # ── FF4 (NaN after hours) ──────────────────────────────────────────────────
    p = m_ff4.params
    df["expected_ff4"] = (
        p.get("const",  0)
        + p.get("MKT_RF", 0) * df["MKT_RF"]
        + p.get("SMB",   0) * df["SMB"]
        + p.get("HML",   0) * df["HML"]
        + p.get("MOM",   0) * df["MOM"]
    )
    df["AR_ff4"]  = df["return"] - df["expected_ff4"]
    df["CAR_ff4"] = df["AR_ff4"].cumsum()

    # ── Simple market-adjusted excess return ───────────────────────────────────
    df["excess_return_simple"] = df["return"] - df["spy_return"]

    output_cols = [
        "price", "return", "spy_return", "excess_return_simple",
        "MKT_RF", "SMB", "HML", "MOM",
        "expected_capm", "AR_capm", "CAR_capm",
        "expected_ff3",  "AR_ff3",  "CAR_ff3",
        "expected_ff4",  "AR_ff4",  "CAR_ff4",
    ]
    return df[[c for c in output_cols if c in df.columns]]


print("AR computation function defined.")

AR computation function defined.


---
## 6. Main Loop — Estimate & Save All Frequencies

In [6]:
# Store model parameters for summary comparison
model_params_store = {}

for freq_name, freq_alias in FREQS.items():
    print(f"\n{'='*65}")
    print(f"  Bar frequency: {freq_name}")
    print(f"{'='*65}")

    # Resample FF factors once per frequency
    ff_bar = resample_ff(ff_1min, freq_alias)

    for symbol, stock_1min in [("GME", gme_1min), ("AMC", amc_1min)]:
        info = STOCKS[symbol]

        # ── Resample stock data ───────────────────────────────────────────────
        bar_data = resample_stock(
            stock_1min, info["price_col"], info["return_col"], freq_alias
        )

        # ── Estimate models at this frequency ─────────────────────────────────
        m_capm, m_ff3, m_ff4 = estimate_models(bar_data, ff_bar, ESTIMATION_END)
        model_params_store[(symbol, freq_name)] = (m_capm, m_ff3, m_ff4)
        model_summary(symbol, freq_name, m_capm, m_ff3, m_ff4)

        # ── Compute AR/CAR columns ─────────────────────────────────────────────
        result = compute_ar_columns(bar_data, ff_bar, m_capm, m_ff3, m_ff4)

        # ── Save CSV ───────────────────────────────────────────────────────────
        out_fname = f"{symbol}-{freq_name}-multi-model-excess-return.csv"
        out_path  = OUTPUT_DIR / out_fname
        result.to_csv(out_path)
        print(f"    → Saved {out_fname}  ({len(result):,} bars)")

print("\nAll CSVs saved.")


  Bar frequency: 1min
  GME @ 1min  (n=18,854 bars in estimation window)
    CAPM : α=+0.000066  β_mkt=1.1300  R²=0.0105  σ=0.005682
    FF3  : α=+0.000055  β_mkt=0.4312  β_smb=0.8179  β_hml=-0.7363  R²=0.0160
    FF4  : α=+0.000053  β_mkt=0.1592  β_smb=0.7944  β_hml=-0.5291  β_mom=0.4511  R²=0.0185


    → Saved GME-1min-multi-model-excess-return.csv  (99,878 bars)
  AMC @ 1min  (n=18,893 bars in estimation window)
    CAPM : α=-0.000054  β_mkt=0.4645  R²=0.0025  σ=0.004789
    FF3  : α=-0.000057  β_mkt=0.1435  β_smb=0.2102  β_hml=-0.1032  R²=0.0016
    FF4  : α=-0.000057  β_mkt=0.1210  β_smb=0.2082  β_hml=-0.0860  β_mom=0.0372  R²=0.0016


    → Saved AMC-1min-multi-model-excess-return.csv  (105,370 bars)

  Bar frequency: 5min
  GME @ 5min  (n=4,364 bars in estimation window)
    CAPM : α=+0.000240  β_mkt=2.4422  R²=0.0293  σ=0.011755
    FF3  : α=+0.000184  β_mkt=0.2897  β_smb=1.6081  β_hml=-1.4858  R²=0.0293
    FF4  : α=+0.000172  β_mkt=-0.7120  β_smb=1.4973  β_hml=-0.7621  β_mom=1.1067  R²=0.0316


    → Saved GME-5min-multi-model-excess-return.csv  (51,744 bars)
  AMC @ 5min  (n=4,365 bars in estimation window)
    CAPM : α=-0.000250  β_mkt=1.4675  R²=0.0159  σ=0.009652
    FF3  : α=-0.000224  β_mkt=-0.9477  β_smb=0.1642  β_hml=-0.6712  R²=0.0228
    FF4  : α=-0.000238  β_mkt=-2.1440  β_smb=0.0320  β_hml=0.1930  β_mom=1.3216  R²=0.0277


    → Saved AMC-5min-multi-model-excess-return.csv  (51,744 bars)

  Bar frequency: 10min
  GME @ 10min  (n=2,216 bars in estimation window)
    CAPM : α=+0.000389  β_mkt=3.0302  R²=0.0453  σ=0.016043
    FF3  : α=+0.000251  β_mkt=0.6119  β_smb=1.8231  β_hml=-1.4476  R²=0.0394
    FF4  : α=+0.000215  β_mkt=-0.8436  β_smb=1.6312  β_hml=-0.3488  β_mom=1.6736  R²=0.0448
    → Saved GME-10min-multi-model-excess-return.csv  (25,872 bars)
  AMC @ 10min  (n=2,216 bars in estimation window)
    CAPM : α=-0.000490  β_mkt=1.7693  R²=0.0253  σ=0.012628
    FF3  : α=-0.000456  β_mkt=-0.4723  β_smb=0.1256  β_hml=-0.3728  R²=0.0064
    FF4  : α=-0.000477  β_mkt=-1.3340  β_smb=0.0120  β_hml=0.2777  β_mom=0.9908  R²=0.0094


    → Saved AMC-10min-multi-model-excess-return.csv  (25,872 bars)

All CSVs saved.


---
## 7. Model Parameter Comparison Across Frequencies

Check that betas are stable across 1-min, 5-min, and 10-min frequencies.
R² should increase slightly at lower frequencies as microstructure noise diminishes.

In [7]:
comparison_rows = []
for (symbol, freq_name), (m_capm, m_ff3, m_ff4) in model_params_store.items():
    for model_name, model in [("CAPM", m_capm), ("FF3", m_ff3), ("FF4", m_ff4)]:
        p = model.params
        comparison_rows.append({
            "Stock":    symbol,
            "Freq":     freq_name,
            "Model":    model_name,
            "alpha":    p.get("const",     0),
            "beta_mkt": p.get("spy_return", p.get("MKT_RF", 0)),
            "beta_smb": p.get("SMB",  np.nan),
            "beta_hml": p.get("HML",  np.nan),
            "beta_mom": p.get("MOM",  np.nan),
            "R2":       model.rsquared,
            "sigma":    np.sqrt(model.mse_resid),
            "n_obs":    int(model.nobs),
        })

comp_df = pd.DataFrame(comparison_rows)
comp_df.style.format({
    "alpha":    "{:.6f}",
    "beta_mkt": "{:.4f}",
    "beta_smb": "{:.4f}",
    "beta_hml": "{:.4f}",
    "beta_mom": "{:.4f}",
    "R2":       "{:.4f}",
    "sigma":    "{:.6f}",
}).set_caption("Model Parameters by Stock, Frequency, and Model")

,Stock,Freq,Model,alpha,beta_mkt,beta_smb,beta_hml,beta_mom,R2,sigma,n_obs
0,GME,1min,CAPM,0.000066,1.1300,nan,nan,nan,0.0105,0.005682,18854
1,GME,1min,FF3,0.000055,0.4312,0.8179,-0.7363,nan,0.0160,0.005666,18854
2,GME,1min,FF4,0.000053,0.1592,0.7944,-0.5291,0.4511,0.0185,0.005659,18854
3,AMC,1min,CAPM,-0.000054,0.4645,nan,nan,nan,0.0025,0.004789,18893
4,AMC,1min,FF3,-0.000057,0.1435,0.2102,-0.1032,nan,0.0016,0.004792,18893
5,AMC,1min,FF4,-0.000057,0.1210,0.2082,-0.0860,0.0372,0.0016,0.004792,18893
6,GME,5min,CAPM,0.000240,2.4422,nan,nan,nan,0.0293,0.011755,4364
7,GME,5min,FF3,0.000184,0.2897,1.6081,-1.4858,nan,0.0293,0.011758,4364
8,GME,5min,FF4,0.000172,-0.7120,1.4973,-0.7621,1.1067,0.0316,0.011745,4364
9,AMC,5min,CAPM,-0.000250,1.4675,nan,nan,nan,0.0159,0.009652,4365


---
## 8. Sample Output

In [8]:
# Preview the 5-min GME CSV around the first outage window
preview_path = OUTPUT_DIR / "GME-5min-multi-model-excess-return.csv"
preview = pd.read_csv(preview_path, parse_dates=["datetime"]).set_index("datetime")

# Show rows around the first outage (Jan 27, 11:00–14:00)
mask = (preview.index >= "2021-01-27 11:00") & (preview.index <= "2021-01-27 14:00")
display_cols = ["price", "return", "excess_return_simple", "AR_capm", "AR_ff3", "AR_ff4",
                "CAR_capm", "CAR_ff3", "CAR_ff4"]
print("GME 5-min — Outage 1 window (Jan 27, 11:00–14:00):")
preview.loc[mask, display_cols].round(6)

GME 5-min — Outage 1 window (Jan 27, 11:00–14:00):


,price,return,excess_return_simple,AR_capm,AR_ff3,AR_ff4,CAR_capm,CAR_ff3,CAR_ff4
datetime,,,,,,,,,
2021-01-27 11:00:00,368.6500,-0.014008,-0.013113,-0.012061,-0.015080,-0.015211,0.033339,0.226831,0.244790
2021-01-27 11:05:00,369.9999,0.002866,0.004053,0.005524,0.001197,0.002004,0.038863,0.228028,0.246794
2021-01-27 11:10:00,350.8175,-0.052699,-0.051908,-0.051005,-0.050418,-0.050566,-0.012142,0.177610,0.196228
2021-01-27 11:15:00,347.4459,-0.010509,-0.009161,-0.007457,-0.007204,-0.007016,-0.019599,0.170407,0.189212
2021-01-27 11:20:00,355.3047,0.022188,0.021792,0.020980,0.022214,0.021407,0.001381,0.192621,0.210619
2021-01-27 11:25:00,363.5799,0.022472,0.022834,0.023116,0.022631,0.021801,0.024497,0.215252,0.232420
2021-01-27 11:30:00,371.5671,0.021816,0.023683,0.026137,0.023914,0.024347,0.050634,0.239166,0.256766
2021-01-27 11:35:00,364.9999,-0.017936,-0.019241,-0.021362,-0.020135,-0.019209,0.029272,0.219031,0.237557
2021-01-27 11:40:00,350.7899,-0.040020,-0.039014,-0.037803,-0.040130,-0.040483,-0.008531,0.178901,0.197074


In [9]:
# Confirm all 6 output files exist
print("Output files:")
for freq_name in FREQS:
    for symbol in ["GME", "AMC"]:
        p = OUTPUT_DIR / f"{symbol}-{freq_name}-multi-model-excess-return.csv"
        size_mb = p.stat().st_size / 1e6 if p.exists() else 0
        status = f"{size_mb:.1f} MB" if p.exists() else "MISSING"
        print(f"  {'✓' if p.exists() else '✗'}  {p.name:<55}  {status}")

Output files:
  ✓  GME-1min-multi-model-excess-return.csv                   24.5 MB
  ✓  AMC-1min-multi-model-excess-return.csv                   25.0 MB
  ✓  GME-5min-multi-model-excess-return.csv                   6.8 MB
  ✓  AMC-5min-multi-model-excess-return.csv                   6.9 MB
  ✓  GME-10min-multi-model-excess-return.csv                  3.4 MB
  ✓  AMC-10min-multi-model-excess-return.csv                  3.5 MB
